# ARENA 1.2: Intro to Mech Interp — nnsight Reimplementation

This notebook reimplements the [ARENA Chapter 1.2 exercises](https://arena-chapter1-transformer-interp.streamlit.app/%5B1.2%5D_Intro_to_Mech_Interp) using **nnsight** instead of TransformerLens.

The model, weights, and exercises are identical — only the intervention/hooking API changes.

**Sections:**
1. Introduction to nnsight (GPT-2)
2. Finding Induction Heads (2-layer attn-only model)
3. Interventions with nnsight (replaces TransformerLens hooks)
4. Reverse-Engineering Induction Circuits

## Setup

In [ ]:
# Cell 1: Install dependencies (%pip ensures install targets the notebook kernel)
%pip install nnsight einops circuitsvis plotly jaxtyping huggingface_hub transformers eindex-callum torch nbformat

In [ ]:
# Cell 2: Imports and NDIF Configuration
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops
import numpy as np
import functools
from dataclasses import dataclass
from typing import Callable, Optional
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
import circuitsvis as cv
from nnsight import NNsight, LanguageModel, CONFIG
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from eindex import eindex
from jaxtyping import Float, Int
from torch import Tensor

# Configure plotly renderer to avoid nbformat detection bug
pio.renderers.default = "plotly_mimetype+notebook"

# Configure NDIF API key for remote execution
CONFIG.set_default_api_key("475b2749-b662-4ab1-9b73-777564e9e252")

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Plotting Utilities

In [ ]:
# Cell 3: Plotting utilities
def to_numpy(tensor):
    if isinstance(tensor, np.ndarray):
        return tensor
    return tensor.detach().cpu().float().numpy()

def imshow(tensor, title="", labels=None, text_auto=False, width=None, height=None,
           x=None, y=None, return_fig=False, **kwargs):
    fig = px.imshow(
        to_numpy(tensor), title=title, labels=labels or {},
        color_continuous_midpoint=0, text_auto=text_auto,
        width=width, height=height, x=x, y=y, **kwargs
    )
    if return_fig:
        return fig
    fig.show()

def line(y_values, title="", x=None, labels=None, width=None, height=None, **kwargs):
    fig = px.line(y=to_numpy(y_values), x=x, title=title, labels=labels or {},
                  width=width, height=height, **kwargs)
    fig.show()

def hist(values, title="", nbins=50, width=None, labels=None, **kwargs):
    fig = px.histogram(x=to_numpy(values) if isinstance(values, (torch.Tensor, np.ndarray)) else values,
                       title=title, nbins=nbins, width=width, labels=labels or {}, **kwargs)
    fig.show()

## Model Architecture

We reimplement the `attn_only_2L_half` model as a plain `nn.Module`, matching the TransformerLens architecture exactly:
- 2 layers, 12 heads, d_model=768, d_head=64
- Attention-only (no MLPs), no LayerNorm
- **Shortformer** positional embeddings: pos_embed is added to Q and K inputs at each layer, but NOT to V or the residual stream

`HookPoint` modules (identity pass-throughs) are placed at key intermediate values so nnsight can intercept them via `.output`.

In [ ]:
# Cell 4: Config, HookPoint, Embed, PosEmbed, Unembed
@dataclass
class ModelConfig:
    d_model: int = 768
    d_head: int = 64
    n_heads: int = 12
    n_layers: int = 2
    n_ctx: int = 2048
    d_vocab: int = 50278
    tokenizer_name: str = "EleutherAI/gpt-neox-20b"


class HookPoint(nn.Module):
    """Identity module. Exists so nnsight can intercept values at this point via .output"""
    def forward(self, x):
        return x


class Embed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_E = nn.Parameter(torch.empty(cfg.d_vocab, cfg.d_model))

    def forward(self, tokens):
        return self.W_E[tokens]


class PosEmbed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_pos = nn.Parameter(torch.empty(cfg.n_ctx, cfg.d_model))

    def forward(self, tokens):
        seq_len = tokens.shape[-1]
        return self.W_pos[:seq_len]


class Unembed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_U = nn.Parameter(torch.empty(cfg.d_model, cfg.d_vocab))
        self.b_U = nn.Parameter(torch.zeros(cfg.d_vocab))

    def forward(self, resid):
        return resid @ self.W_U + self.b_U

In [ ]:
# Cell 5: Attention module with hook points
class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_K = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_V = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_O = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_head, cfg.d_model))
        self.b_Q = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_K = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_V = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_O = nn.Parameter(torch.zeros(cfg.d_model))
        self.register_buffer("mask", torch.tril(torch.ones(cfg.n_ctx, cfg.n_ctx, dtype=torch.bool)))
        self.register_buffer("IGNORE", torch.tensor(float("-inf")))
        # Hook points for nnsight interception
        self.hook_q = HookPoint()
        self.hook_k = HookPoint()
        self.hook_v = HookPoint()
        self.hook_pattern = HookPoint()
        self.hook_z = HookPoint()
        self.hook_result = HookPoint()

    def forward(self, resid_pre, shortformer_pos_embed):
        # Shortformer: add pos_embed to Q and K inputs only
        q_input = resid_pre + shortformer_pos_embed
        k_input = resid_pre + shortformer_pos_embed
        v_input = resid_pre  # V does NOT get positional embeddings

        # Project to Q, K, V: [batch, pos, n_heads, d_head]
        q = einops.einsum(q_input, self.W_Q, "b p m, h m d -> b p h d") + self.b_Q
        k = einops.einsum(k_input, self.W_K, "b p m, h m d -> b p h d") + self.b_K
        v = einops.einsum(v_input, self.W_V, "b p m, h m d -> b p h d") + self.b_V

        q = self.hook_q(q)
        k = self.hook_k(k)
        v = self.hook_v(v)

        # Attention scores: [batch, n_heads, q_pos, k_pos]
        attn_scores = einops.einsum(
            q, k, "b qp h d, b kp h d -> b h qp kp"
        ) / (self.cfg.d_head ** 0.5)

        # Causal mask
        seq_len = resid_pre.shape[1]
        attn_scores = torch.where(
            self.mask[:seq_len, :seq_len], attn_scores, self.IGNORE
        )

        # Softmax
        pattern = F.softmax(attn_scores, dim=-1)
        pattern = torch.where(torch.isnan(pattern), torch.zeros_like(pattern), pattern)
        pattern = self.hook_pattern(pattern)  # [batch, n_heads, q_pos, k_pos]

        # Weighted values: [batch, pos, n_heads, d_head]
        z = einops.einsum(pattern, v, "b h qp kp, b kp h d -> b qp h d")
        z = self.hook_z(z)

        # Per-head output: [batch, pos, n_heads, d_model]
        result = einops.einsum(z, self.W_O, "b p h d, h d m -> b p h m")
        result = self.hook_result(result)

        # Sum heads + bias: [batch, pos, d_model]
        attn_out = result.sum(dim=2) + self.b_O
        return attn_out

In [ ]:
# Cell 6: Block and full AttnOnly2L model
class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.attn = Attention(cfg)

    def forward(self, resid_pre, shortformer_pos_embed):
        attn_out = self.attn(resid_pre, shortformer_pos_embed)
        return resid_pre + attn_out


class AttnOnly2L(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.unembed = Unembed(cfg)

    def forward(self, tokens):
        token_embed = self.embed(tokens)
        pos_embed = self.pos_embed(tokens)
        # Shortformer: residual starts as just token embeddings (no pos_embed)
        resid = token_embed
        for block in self.blocks:
            resid = block(resid, pos_embed)
        return self.unembed(resid)

    @classmethod
    def from_pretrained(cls, device="cpu"):
        cfg = ModelConfig()
        model = cls(cfg)
        weights_path = hf_hub_download(
            repo_id="callummcdougall/attn_only_2L_half",
            filename="attn_only_2L_half.pth",
        )
        pretrained = torch.load(weights_path, map_location=device, weights_only=True)
        # Our parameter names match TransformerLens keys exactly.
        # strict=False to skip buffers (mask, IGNORE) which we recreate.
        missing, unexpected = model.load_state_dict(pretrained, strict=False)
        print(f"Missing keys (expected - hook points have no params): {missing}")
        print(f"Unexpected keys (expected - checkpoint buffers): {unexpected}")
        return model.to(device)

## FactoredMatrix Utility

A lightweight reimplementation of TransformerLens's `FactoredMatrix` for efficient circuit analysis without materializing large (d_vocab × d_vocab) matrices.

In [ ]:
# Cell 7: FactoredMatrix utility
class FactoredMatrix:
    def __init__(self, A: Tensor, B: Tensor):
        self.A = A
        self.B = B

    @property
    def AB(self) -> Tensor:
        return self.A @ self.B

    @property
    def T(self) -> "FactoredMatrix":
        return FactoredMatrix(self.B.mT, self.A.mT)

    @property
    def shape(self):
        return (*self.A.shape[:-1], self.B.shape[-1])

    def __matmul__(self, other):
        if isinstance(other, FactoredMatrix):
            return FactoredMatrix(self.A, self.B @ other.A @ other.B)
        elif isinstance(other, Tensor):
            return FactoredMatrix(self.A, self.B @ other)
        return NotImplemented

    def __rmatmul__(self, other):
        if isinstance(other, Tensor):
            return FactoredMatrix(other @ self.A, self.B)
        return NotImplemented

    def __getitem__(self, idx):
        if isinstance(idx, tuple) and len(idx) == 2:
            # Grid indexing: rows from A, cols from B
            row_idx, col_idx = idx
            return FactoredMatrix(self.A[..., row_idx, :], self.B[..., :, col_idx])
        elif isinstance(idx, Tensor) and self.A.dim() == 2:
            # Row selection on a 2D factored matrix
            return FactoredMatrix(self.A[idx], self.B)
        else:
            # Leading/batch dimension indexing
            return FactoredMatrix(self.A[idx], self.B[idx])

    def norm(self) -> Tensor:
        """Efficient Frobenius norm: ||AB||_F without materializing AB.
        Uses: ||AB||_F^2 = tr(B^T A^T A B) = sum_{ij} (A^T A)_{ij} * (B B^T)_{ij}
        """
        ATA = self.A.mT @ self.A
        BBT = self.B @ self.B.mT
        return (ATA * BBT).sum((-2, -1)).sqrt()

    def get_corner(self, n: int = 10) -> Tensor:
        return self.A[..., :n, :] @ self.B[..., :, :n]

---
## Section 1: Introduction to nnsight

We use GPT-2 Small to introduce nnsight's core API:
- `LanguageModel` wraps HuggingFace models
- `model.trace(input)` defines an intervention context
- `.output.save()` captures module outputs

### Loading and Running Models

In [ ]:
# Cell 8: Load GPT-2 with nnsight
# Use device_map="auto" for automatic device placement
# For remote execution, the model loads on 'meta' device and runs on NDIF servers
gpt2 = LanguageModel("openai-community/gpt2", device_map="auto")

In [ ]:
# Cell 9: Run GPT-2, compute loss
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

# In nnsight, we run the model inside a trace context and save the outputs we need
with gpt2.trace(model_description_text):
    logits = gpt2.lm_head.output.save()

# Compute loss manually (TransformerLens had return_type="loss")
tokens = gpt2.tokenizer(model_description_text, return_tensors="pt")["input_ids"].to(device)
loss = F.cross_entropy(logits[0, :-1], tokens[0, 1:])
print("Model loss:", loss.item())

### Tokenization

In [ ]:
# Cell 10: Tokenization basics
tokenizer_gpt2 = gpt2.tokenizer

print("Tokens for 'gpt2':", tokenizer_gpt2.tokenize("gpt2"))
print("Token IDs:", tokenizer_gpt2.encode("gpt2"))
print("Decoded:", tokenizer_gpt2.decode([50256, 70, 457, 17]))

### Exercise: How many tokens does the model guess correctly?

In [ ]:
# Cell 11: Model accuracy exercise
with gpt2.trace(model_description_text):
    logits = gpt2.lm_head.output.save()

tokens = gpt2.tokenizer(model_description_text, return_tensors="pt")["input_ids"].squeeze()
prediction = logits[0].argmax(dim=-1).squeeze()[:-1]
true_tokens = tokens[1:].to(device)
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {tokenizer_gpt2.batch_decode(prediction[is_correct.cpu()])}")

### Caching Activations

In TransformerLens: `logits, cache = model.run_with_cache(tokens)`  
In nnsight: save whatever you need inside the trace context.

In [ ]:
# Cell 12: Cache activations via nnsight trace
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2.tokenizer(gpt2_text, return_tensors="pt")["input_ids"].to(device)

with gpt2.trace(gpt2_tokens):
    # Save Q, K, V projections from layer 0 attention
    # GPT-2 uses a fused c_attn that outputs [batch, seq, 3*d_model] for Q,K,V
    qkv_l0 = gpt2.transformer.h[0].attn.c_attn.output.save()
    gpt2_logits = gpt2.lm_head.output.save()

print(f"QKV shape: {qkv_l0.shape}")
print(f"Logits shape: {gpt2_logits.shape}")

### Exercise: Verify attention patterns from Q and K

In [ ]:
# Cell 13: Verify attention patterns from Q and K
d_model_gpt2 = 768
n_heads_gpt2 = 12
d_head_gpt2 = 64

q, k, v = qkv_l0[0].split(d_model_gpt2, dim=-1)  # each [seq, d_model]
seq_len_gpt2 = q.shape[0]

# Reshape to [n_heads, seq, d_head]
q = q.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)
k = k.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)

# Compute attention pattern
attn_scores = torch.einsum("hqd,hkd->hqk", q, k) / (d_head_gpt2 ** 0.5)
causal_mask = torch.triu(torch.ones(seq_len_gpt2, seq_len_gpt2, dtype=torch.bool, device=device), diagonal=1)
attn_scores.masked_fill_(causal_mask, -1e9)
layer0_pattern = F.softmax(attn_scores, dim=-1)

print(f"Attention pattern shape: {layer0_pattern.shape}")
print("Attention pattern computed successfully from Q and K!")

### Visualizing Attention Heads

In [ ]:
# Cell 14: Visualize attention heads with circuitsvis
gpt2_str_tokens = gpt2.tokenizer.tokenize(gpt2_text)
# Prepend the BOS/first token that tokenize might not include
full_str_tokens = [gpt2.tokenizer.decode(t) for t in gpt2_tokens[0].tolist()]

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_heads(
        tokens=full_str_tokens,
        attention=layer0_pattern.cpu(),
        attention_head_names=[f"L0H{i}" for i in range(n_heads_gpt2)],
    )
)

---
## Section 2: Finding Induction Heads

### Loading the 2-Layer Attention-Only Model

We load the custom `attn_only_2L_half` model as our plain `nn.Module` and wrap it with `NNsight`.

In [ ]:
# Cell 15: Load 2-layer model, wrap with NNsight
cfg = ModelConfig()
raw_model = AttnOnly2L.from_pretrained(device=device)
model = NNsight(raw_model)

# Load tokenizer separately (NNsight base class doesn't handle tokenization)
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)

print(f"Model loaded. Config: {cfg}")

In [ ]:
# Cell 16: Run model, save attention patterns
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

tokens = tokenizer(text, return_tensors="pt")["input_ids"].to(device)

# Initialize variables BEFORE trace (nnsight scope quirk)
patterns = {}
logits_saved = None

# IMPORTANT: Access modules in forward-pass order (intermediate first, then final output)
with model.trace(tokens):
    for layer in range(cfg.n_layers):
        patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()
    logits_saved = model.output.save()

print(f"Logits shape: {logits_saved.shape}")
print(f"Attention pattern shape (layer 0): {patterns[0].shape}")

In [ ]:
# Cell 17: Visualize 2-layer model attention patterns
str_tokens = [tokenizer.decode(t) for t in tokens[0].tolist()]

for layer in range(cfg.n_layers):
    attention_pattern = patterns[layer].squeeze(0)  # remove batch dim
    display(cv.attention.attention_heads(
        tokens=str_tokens, attention=attention_pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(cfg.n_heads)],
    ))

### Writing Detectors for Attention Patterns

These detectors work on saved attention patterns (not TransformerLens caches).

In [ ]:
# Cell 18: Attention pattern detectors
def current_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects current-token heads (strong diagonal pattern)."""
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)  # [n_heads, q, k]
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def prev_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects previous-token heads (strong sub-diagonal pattern)."""
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def first_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects first-token heads (strong attention to position 0)."""
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head][:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


print("Heads attending to current token  =", ", ".join(current_attn_detector(patterns)))
print("Heads attending to previous token =", ", ".join(prev_attn_detector(patterns)))
print("Heads attending to first token    =", ", ".join(first_attn_detector(patterns)))

### Induction Heads

Induction heads implement the algorithm: if the model has seen `[A][B]...[A]`, predict `[B]`. This requires:
1. A **previous-token head** (layer 0) that writes "I was preceded by token X" into the residual stream
2. An **induction head** (layer 1) that attends to the token *after* a previous occurrence of the current token, then copies what comes next

This can't form in a 1-layer model because attention scores depend only on the current token's query and each key — you can't look at *adjacent* tokens without composition across layers.

In [ ]:
# Cell 19: Generate repeated tokens and log prob helpers
def generate_repeated_tokens(seq_len: int, batch_size: int = 1) -> Int[Tensor, "batch full_seq"]:
    """Generates [BOS, rand_tokens, rand_tokens] sequences."""
    torch.manual_seed(0)
    prefix = (torch.ones(batch_size, 1) * tokenizer.bos_token_id).long()
    half = torch.randint(0, cfg.d_vocab, (batch_size, seq_len), dtype=torch.int64)
    return torch.cat([prefix, half, half], dim=-1).to(device)


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"],
    tokens: Int[Tensor, "batch posn"],
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    return eindex(logprobs, tokens, "b s [b s+1]")

In [ ]:
# Cell 20: Run on repeated tokens, compare first vs second half
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

# Initialize variables BEFORE trace (nnsight scope quirk)
rep_patterns = {}
rep_embed = None
rep_pos_embed = None
rep_logits = None

# IMPORTANT: Access modules in forward-pass order
with model.trace(rep_tokens):
    rep_embed = model.embed.output.save()
    rep_pos_embed = model.pos_embed.output.save()
    for layer in range(cfg.n_layers):
        rep_patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()
    rep_logits = model.output.save()

log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

In [ ]:
# Cell 21: Plot per-token loss
rep_str = [tokenizer.decode(t) for t in rep_tokens[0].tolist()]

fig = px.line(y=to_numpy(-log_probs), title="Per-token log prob on repeated sequence")
fig.add_vrect(x0=seq_len-0.5, x1=2*seq_len-0.5, fillcolor="green", opacity=0.1,
              annotation_text="Second half (should be predicted)")
fig.show()

In [ ]:
# Cell 22: Visualize attention on repeated tokens
for layer in range(cfg.n_layers):
    attention_pattern = rep_patterns[layer].squeeze(0)
    display(cv.attention.attention_heads(
        tokens=rep_str, attention=attention_pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(cfg.n_heads)],
    ))

### Exercise: Induction Head Detector

In [ ]:
# Cell 23: Induction head detector
def induction_attn_detector(patterns: dict[int, Tensor], seq_len: int) -> list[str]:
    """Detects induction heads by looking for the characteristic offset-diagonal stripe.
    For repeated tokens [BOS, r1..r_N, r1..r_N], induction heads attend from
    position i (second half) to position i - seq_len + 1 (first half).
    """
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)  # [n_heads, q, k]
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


print("Induction heads =", ", ".join(induction_attn_detector(rep_patterns, seq_len)))

---
## Section 3: Interventions with nnsight

TransformerLens uses `model.run_with_hooks()` with hook functions.  
nnsight replaces this with the trace context — you read and write to module outputs directly.

### Computing Induction Scores with nnsight

In [ ]:
# Cell 24: Induction scores via nnsight trace
seq_len = 50
batch_size = 10
rep_tokens_10 = generate_repeated_tokens(seq_len, batch_size)

# Initialize variables BEFORE trace (nnsight scope quirk)
saved_patterns = {}

# Save all attention patterns in a single trace
with model.trace(rep_tokens_10):
    for layer in range(cfg.n_layers):
        saved_patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()

# Compute induction scores after the trace
induction_score_store = torch.zeros(cfg.n_layers, cfg.n_heads, device=device)

for layer in range(cfg.n_layers):
    pattern = saved_patterns[layer]  # [batch, n_heads, q_pos, k_pos]
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(induction_stripe, "batch head pos -> head", "mean")
    induction_score_store[layer] = induction_score

imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".2f", width=900, height=350,
)

### Finding Induction Heads in GPT-2 Small

For GPT-2, we compute attention patterns from the fused Q/K/V projection (`c_attn`).

In [ ]:
# Cell 25: Helper to compute GPT-2 attention patterns from fused QKV
def compute_gpt2_attention_patterns(qkv: Tensor) -> Tensor:
    """Compute attention patterns from GPT-2's fused c_attn output.
    qkv: [batch, seq, 3*d_model] -> returns [batch, n_heads, seq, seq]
    """
    d_model, n_heads, d_head = 768, 12, 64
    q, k, _ = qkv.split(d_model, dim=-1)
    batch, seq = q.shape[:2]
    q = q.view(batch, seq, n_heads, d_head).permute(0, 2, 1, 3)
    k = k.view(batch, seq, n_heads, d_head).permute(0, 2, 1, 3)
    scores = torch.einsum("bhqd,bhkd->bhqk", q, k) / (d_head ** 0.5)
    causal_mask = torch.triu(torch.ones(seq, seq, dtype=torch.bool, device=scores.device), diagonal=1)
    scores.masked_fill_(causal_mask, -1e9)
    return F.softmax(scores, dim=-1)

In [ ]:
# Cell 26: Find induction heads in GPT-2 Small
seq_len = 50
batch_size = 10
rep_tokens_gpt2 = generate_repeated_tokens(seq_len, batch_size)

# Initialize variables BEFORE trace (nnsight scope quirk)
qkv_all_layers = {}

# Save c_attn outputs for all 12 layers in one trace
with gpt2.trace(rep_tokens_gpt2):
    for layer in range(12):
        qkv_all_layers[layer] = gpt2.transformer.h[layer].attn.c_attn.output.save()

# Compute induction scores
induction_score_store_gpt2 = torch.zeros(12, 12, device=device)

for layer in range(12):
    pattern = compute_gpt2_attention_patterns(qkv_all_layers[layer])
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(induction_stripe, "batch head pos -> head", "mean")
    induction_score_store_gpt2[layer] = induction_score

imshow(
    induction_score_store_gpt2,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head (GPT-2 Small)",
    text_auto=".1f", width=700, height=500,
)

In [ ]:
# Cell 27: Visualize GPT-2 induction head layers
rep_tokens_single = generate_repeated_tokens(seq_len, batch_size=1)
rep_str_gpt2 = [gpt2.tokenizer.decode(t) for t in rep_tokens_single[0].tolist()]

# Initialize variables BEFORE trace (nnsight scope quirk)
qkv_viz = {}

with gpt2.trace(rep_tokens_single):
    for layer in [5, 6, 7]:
        qkv_viz[layer] = gpt2.transformer.h[layer].attn.c_attn.output.save()

for layer in [5, 6, 7]:
    pattern = compute_gpt2_attention_patterns(qkv_viz[layer]).squeeze(0)  # [n_heads, seq, seq]
    print(f"Layer {layer}:")
    display(cv.attention.attention_heads(
        tokens=rep_str_gpt2, attention=pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(12)],
    ))

### Direct Logit Attribution

Since the residual stream is a sum of component outputs, we can decompose the final logit into contributions from:
- The embedding (direct path)
- Each attention head in layer 0
- Each attention head in layer 1

In [ ]:
# Cell 28: Logit attribution function
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Returns logit attributions of shape (seq-1, 1 + 2*n_heads).
    Columns: [direct_path, L0H0, ..., L0H11, L1H0, ..., L1H11]
    """
    W_U_correct = W_U[:, tokens[1:]]

    direct = einops.einsum(W_U_correct, embed[:-1], "emb seq, seq emb -> seq")
    l1_attr = einops.einsum(W_U_correct, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    l2_attr = einops.einsum(W_U_correct, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead")

    return torch.cat([direct.unsqueeze(-1), l1_attr, l2_attr], dim=-1)

In [ ]:
# Cell 29: Compute and verify logit attribution
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
tokens = tokenizer(text, return_tensors="pt")["input_ids"].to(device)

# Initialize variables BEFORE trace (nnsight scope quirk)
embed = None
l1_results = None
l2_results = None
logits_saved = None

# IMPORTANT: Access modules in forward-pass order
with model.trace(tokens):
    embed = model.embed.output.save()
    l1_results = model.blocks[0].attn.hook_result.output.save()
    l2_results = model.blocks[1].attn.hook_result.output.save()
    logits_saved = model.output.save()

# Remove batch dim
embed_sq = embed.squeeze(0)
l1_results_sq = l1_results.squeeze(0)
l2_results_sq = l2_results.squeeze(0)

W_U = raw_model.unembed.W_U

logit_attr = logit_attribution(embed_sq, l1_results_sq, l2_results_sq, W_U, tokens.squeeze())

# Verify: sum of attributions should equal actual logits for correct tokens
correct_token_logits = logits_saved[0, torch.arange(len(tokens[0]) - 1), tokens[0, 1:]]
torch.testing.assert_close(logit_attr.sum(1), correct_token_logits, atol=1e-2, rtol=0)
print("Logit attribution test passed!")

In [ ]:
# Cell 30: Plot logit attribution (demo prompt)
str_tokens = [tokenizer.decode(t) for t in tokens[0].tolist()]
component_labels = ["Direct"] + [f"L0H{i}" for i in range(cfg.n_heads)] + [f"L1H{i}" for i in range(cfg.n_heads)]

imshow(
    logit_attr.T,
    x=[str_tokens[i] for i in range(1, len(str_tokens))],
    y=component_labels,
    labels={"x": "Token", "y": "Component", "color": "Logit contribution"},
    title="Logit attribution (demo prompt)",
    width=1200, height=500,
)

In [ ]:
# Cell 31: Logit attribution on induction prompt
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

# Initialize variables BEFORE trace (nnsight scope quirk)
rep_embed = None
rep_l1_results = None
rep_l2_results = None
rep_logits = None

# IMPORTANT: Access modules in forward-pass order
with model.trace(rep_tokens):
    rep_embed = model.embed.output.save()
    rep_l1_results = model.blocks[0].attn.hook_result.output.save()
    rep_l2_results = model.blocks[1].attn.hook_result.output.save()
    rep_logits = model.output.save()

logit_attr_rep = logit_attribution(
    rep_embed.squeeze(0), rep_l1_results.squeeze(0), rep_l2_results.squeeze(0),
    W_U, rep_tokens.squeeze(),
)

rep_str = [tokenizer.decode(t) for t in rep_tokens[0].tolist()]
imshow(
    logit_attr_rep.T,
    x=[rep_str[i] for i in range(1, len(rep_str))],
    y=component_labels,
    labels={"x": "Token", "y": "Component", "color": "Logit contribution"},
    title="Logit attribution (random induction prompt)",
    width=1200, height=500,
)

### Ablations

In TransformerLens, you write a hook function and use `run_with_hooks`.  
In nnsight, you directly modify activations inside the trace context.

#### Zero Ablation

In [ ]:
# Cell 32: Ablation scoring function
def get_ablation_scores(
    model: NNsight,
    tokens: Int[Tensor, "batch seq"],
    ablation_type: str = "zero",
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns increase in cross-entropy loss from ablating each head.
    ablation_type: 'zero' (set to 0) or 'mean' (replace with mean across positions).
    """
    seq_len = (tokens.shape[1] - 1) // 2
    ablation_scores = torch.zeros(cfg.n_layers, cfg.n_heads, device=device)

    # Baseline loss (no ablation)
    with model.trace(tokens):
        clean_logits = model.output.save()
    loss_no_ablation = -get_log_probs(clean_logits, tokens)[:, -(seq_len - 1):].mean()

    for layer in tqdm(range(cfg.n_layers)):
        for head in range(cfg.n_heads):
            with model.trace(tokens):
                # Intervene: modify z (pre-projection head output) at this layer
                # Use direct modification on the proxy for cleaner nnsight pattern
                if ablation_type == "zero":
                    model.blocks[layer].attn.hook_z.output[:, :, head, :] = 0.0
                elif ablation_type == "mean":
                    # For mean ablation, we need to compute mean and assign
                    z_head = model.blocks[layer].attn.hook_z.output[:, :, head, :]
                    mean_z = z_head.mean(dim=1, keepdim=True)
                    model.blocks[layer].attn.hook_z.output[:, :, head, :] = mean_z
                ablated_logits = model.output.save()

            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1):].mean()
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores

In [ ]:
# Cell 33: Zero ablation scores
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

ablation_scores = get_ablation_scores(model, rep_tokens, ablation_type="zero")

imshow(
    ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Loss diff"},
    title="Loss Difference After Zero-Ablating Heads",
    text_auto=".2f", width=900, height=350,
)

#### Mean Ablation

In [ ]:
# Cell 34: Mean ablation scores
rep_tokens_batch = generate_repeated_tokens(seq_len=50, batch_size=10)

mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_type="mean")

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Loss diff"},
    title="Loss Difference After Mean-Ablating Heads",
    text_auto=".2f", width=900, height=350,
)

---
## Section 4: Reverse-Engineering Induction Circuits

### OV Circuits

The OV circuit determines **what a head writes** when it attends to a token.  
Full OV circuit: `W_E @ W_V @ W_O @ W_U` — maps input tokens to output logits.

In [ ]:
# Cell 35: OV circuit for copying head
head_index = 4
layer = 1

W_V = raw_model.blocks[layer].attn.W_V[head_index]  # [d_model, d_head]
W_O = raw_model.blocks[layer].attn.W_O[head_index]  # [d_head, d_model]
W_E = raw_model.embed.W_E                            # [d_vocab, d_model]
W_U = raw_model.unembed.W_U                          # [d_model, d_vocab]

OV_circuit = FactoredMatrix(W_V, W_O)
full_OV_circuit = W_E @ OV_circuit @ W_U

print(f"Full OV circuit shape: {full_OV_circuit.shape}")

In [ ]:
# Cell 36: Visualize OV circuit diagonal
indices = torch.randint(0, cfg.d_vocab, (200,))
full_OV_circuit_sample = full_OV_circuit[indices, indices].AB

imshow(
    full_OV_circuit_sample,
    labels={"x": "Logits on output token", "y": "Input token"},
    title="Full OV circuit for copying head (L1H4)",
    width=700, height=600,
)

In [ ]:
# Cell 37: Top-1 accuracy of OV circuit
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """Fraction of tokens where the argmax logit equals the input token (diagonal)."""
    total = 0
    for indices in torch.split(torch.arange(full_OV_circuit.shape[0], device=device), batch_size):
        AB_slice = full_OV_circuit[indices].AB
        total += (AB_slice.argmax(dim=1) == indices).float().sum().item()
    return total / full_OV_circuit.shape[0]


print(f"Fraction of time best logit is on diagonal (L1H4): {top_1_acc(full_OV_circuit):.4f}")

In [ ]:
# Cell 38: Combined OV circuit for both induction heads
W_O_both = einops.rearrange(
    raw_model.blocks[1].attn.W_O[[4, 10]], "head d_head d_model -> (head d_head) d_model"
)
W_V_both = einops.rearrange(
    raw_model.blocks[1].attn.W_V[[4, 10]], "head d_model d_head -> d_model (head d_head)"
)

W_E = raw_model.embed.W_E
W_U = raw_model.unembed.W_U
W_OV_eff = W_E @ FactoredMatrix(W_V_both, W_O_both) @ W_U

print(f"Combined OV accuracy: {top_1_acc(W_OV_eff):.4f}")

### QK Circuits

The QK circuit determines **what a head attends to**.  
For positional QK: `W_pos @ W_Q @ W_K^T @ W_pos^T` — which positions attend to which.

In [ ]:
# Cell 39: QK circuit for previous-token head
layer = 0
head_index = 7

W_pos = raw_model.pos_embed.W_pos
W_Q_head = raw_model.blocks[layer].attn.W_Q[head_index]  # [d_model, d_head]
W_K_head = raw_model.blocks[layer].attn.W_K[head_index]  # [d_model, d_head]

W_QK = W_Q_head @ W_K_head.T  # [d_model, d_model]
pos_by_pos_scores = W_pos @ W_QK @ W_pos.T  # [n_ctx, n_ctx]

# Mask, scale, softmax
mask = torch.tril(torch.ones_like(pos_by_pos_scores)).bool()
pos_by_pos_pattern = torch.where(
    mask, pos_by_pos_scores / cfg.d_head ** 0.5, torch.tensor(-1e6, device=device)
).softmax(-1)

print(f"Avg lower-diagonal value: {pos_by_pos_pattern.diag(-1).mean():.4f}")

imshow(
    pos_by_pos_pattern[:200, :200],
    labels={"x": "Key", "y": "Query"},
    title="Attention patterns for prev-token QK circuit (L0H7), first 200 positions",
    width=700, height=600,
)

### Decomposing QK Input

The input to layer 1's Q and K projections (in shortformer mode) is `resid_pre_1 + pos_embed`.  
We decompose this as: `embed + pos_embed + sum(layer0_head_results)`.

In [ ]:
# Cell 40: Save activations for QK decomposition
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

# Initialize variables BEFORE trace (nnsight scope quirk)
embed_saved = None
pos_embed_saved = None
l0_result = None
q_saved = None
k_saved = None

with model.trace(rep_tokens):
    embed_saved = model.embed.output.save()        # [1, seq, d_model]
    pos_embed_saved = model.pos_embed.output.save() # [seq, d_model]
    l0_result = model.blocks[0].attn.hook_result.output.save()  # [1, seq, n_heads, d_model]
    q_saved = model.blocks[1].attn.hook_q.output.save()  # [1, seq, n_heads, d_head]
    k_saved = model.blocks[1].attn.hook_k.output.save()  # [1, seq, n_heads, d_head]

In [ ]:
# Cell 41: QK decomposition functions
def decompose_qk_input(
    embed: Tensor, pos_embed: Tensor, l0_result: Tensor,
) -> Float[Tensor, "n_components seq d_model"]:
    """Decompose the QK input into: [embed, pos_embed, head0_result, ..., head11_result]"""
    embed_sq = embed.squeeze(0)       # [seq, d_model]
    l0_per_head = l0_result.squeeze(0).permute(1, 0, 2)  # [n_heads, seq, d_model]
    return torch.cat([
        embed_sq.unsqueeze(0),        # [1, seq, d_model]
        pos_embed.unsqueeze(0),        # [1, seq, d_model]
        l0_per_head,                   # [n_heads, seq, d_model]
    ], dim=0)


def decompose_q(
    decomposed_input: Float[Tensor, "comp seq d_model"],
    ind_head_index: int,
) -> Float[Tensor, "comp seq d_head"]:
    W_Q = raw_model.blocks[1].attn.W_Q[ind_head_index]
    return einops.einsum(decomposed_input, W_Q, "n seq d_model, d_model d_head -> n seq d_head")


def decompose_k(
    decomposed_input: Float[Tensor, "comp seq d_model"],
    ind_head_index: int,
) -> Float[Tensor, "comp seq d_head"]:
    W_K = raw_model.blocks[1].attn.W_K[ind_head_index]
    return einops.einsum(decomposed_input, W_K, "n seq d_model, d_model d_head -> n seq d_head")

In [ ]:
# Cell 42: Verify QK decomposition
ind_head_index = 4

decomposed_qk_input = decompose_qk_input(embed_saved, pos_embed_saved, l0_result)
decomposed_q_vals = decompose_q(decomposed_qk_input, ind_head_index)
decomposed_k_vals = decompose_k(decomposed_qk_input, ind_head_index)

# Verify: sum of decomposed Q should match actual Q for this head
torch.testing.assert_close(
    decomposed_q_vals.sum(0), q_saved.squeeze(0)[:, ind_head_index],
    rtol=0.01, atol=0.001,
)
torch.testing.assert_close(
    decomposed_k_vals.sum(0), k_saved.squeeze(0)[:, ind_head_index],
    rtol=0.01, atol=0.01,
)
print("Decomposition tests passed!")

In [ ]:
# Cell 43: Plot QK component norms
component_labels = ["Embed", "PosEmbed"] + [f"0.{h}" for h in range(cfg.n_heads)]

for decomposed_input, name in [(decomposed_q_vals, "query"), (decomposed_k_vals, "key")]:
    imshow(
        decomposed_input.pow(2).sum(-1),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800, height=400,
    )

### Decomposing Attention Scores

In [ ]:
# Cell 44: Decompose attention scores
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    return einops.einsum(
        decomposed_q, decomposed_k,
        "qc qp d, kc kp d -> qc kc qp kp",
    ) / (cfg.d_head ** 0.5)


decomposed_scores = decompose_attn_scores(decomposed_q_vals, decomposed_k_vals)

In [ ]:
# Cell 45: Visualize dominant (Embed, 0.7) contribution and std devs
q_label, k_label = "Embed", "0.7"
qi, ki = component_labels.index(q_label), component_labels.index(k_label)

imshow(
    torch.tril(decomposed_scores[qi, ki]),
    title=f"Attention score contributions: query={q_label}, key={k_label}",
    width=700,
)

# Std dev over positions shows which (q_comp, k_comp) pairs matter most
decomposed_stds = einops.reduce(
    decomposed_scores, "qc kc qp kp -> qc kc", torch.std,
)
imshow(
    decomposed_stds,
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attention score contributions",
    x=component_labels, y=component_labels,
    width=700,
)

### K-Composition: The Full Induction Circuit

The induction head's key input comes from the previous-token head's OV circuit.  
Full K-composition circuit: `(W_E @ W_Q)` × `(W_E @ W_V @ W_O @ W_K)^T`

In [ ]:
# Cell 46: K-composition full circuit
def find_K_comp_full_circuit(
    prev_token_head_index: int, ind_head_index: int,
) -> FactoredMatrix:
    W_E = raw_model.embed.W_E
    W_Q = raw_model.blocks[1].attn.W_Q[ind_head_index]
    W_K = raw_model.blocks[1].attn.W_K[ind_head_index]
    W_V = raw_model.blocks[0].attn.W_V[prev_token_head_index]
    W_O = raw_model.blocks[0].attn.W_O[prev_token_head_index]

    Q = W_E @ W_Q        # [d_vocab, d_head]
    K = W_E @ W_V @ W_O @ W_K  # [d_vocab, d_head]
    return FactoredMatrix(Q, K.T)


K_comp_circuit = find_K_comp_full_circuit(prev_token_head_index=7, ind_head_index=4)
print(f"K-comp circuit shape: {K_comp_circuit.shape}")
print(f"Token frac where max-activating key = same token: {top_1_acc(K_comp_circuit.T):.4f}")

### Composition Scores

Measures how strongly two heads compose: `||W_A @ W_B|| / (||W_A|| * ||W_B||)`

In [ ]:
# Cell 47: Composition score function
def get_comp_score(
    W_A: Float[Tensor, "in_A out_A"],
    W_B: Float[Tensor, "out_A out_B"],
) -> float:
    W_A_norm = W_A.pow(2).sum().sqrt()
    W_B_norm = W_B.pow(2).sum().sqrt()
    W_AB_norm = (W_A @ W_B).pow(2).sum().sqrt()
    return (W_AB_norm / (W_A_norm * W_B_norm)).item()

In [ ]:
# Cell 48: Q, K, V composition scores between all L0-L1 head pairs
def get_W_OV(layer, head):
    return raw_model.blocks[layer].attn.W_V[head] @ raw_model.blocks[layer].attn.W_O[head]

def get_W_QK(layer, head):
    return raw_model.blocks[layer].attn.W_Q[head] @ raw_model.blocks[layer].attn.W_K[head].T

composition_scores = {
    "Q": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
    "K": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
    "V": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
}

for i in tqdm(range(cfg.n_heads)):
    W_OV_0i = get_W_OV(0, i)
    for j in range(cfg.n_heads):
        W_QK_1j = get_W_QK(1, j)
        composition_scores["Q"][i, j] = get_comp_score(W_OV_0i, W_QK_1j)
        composition_scores["K"][i, j] = get_comp_score(W_OV_0i, W_QK_1j.T)
        composition_scores["V"][i, j] = get_comp_score(W_OV_0i, get_W_OV(1, j))

for comp_type in ["Q", "K", "V"]:
    imshow(
        composition_scores[comp_type],
        labels={"x": "L1 Head", "y": "L0 Head", "color": "Score"},
        title=f"{comp_type} Composition Scores",
        text_auto=".2f", width=700, height=600,
    )

### Random Baseline for Composition Scores

In [ ]:
# Cell 49: Random baseline for composition scores
def generate_single_random_comp_score() -> float:
    W_A_left = torch.empty(cfg.d_model, cfg.d_head)
    W_B_left = torch.empty(cfg.d_model, cfg.d_head)
    W_A_right = torch.empty(cfg.d_model, cfg.d_head)
    W_B_right = torch.empty(cfg.d_model, cfg.d_head)
    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T
    return get_comp_score(W_A, W_B)


n_samples = 300
comp_scores_baseline = np.array([generate_single_random_comp_score() for _ in tqdm(range(n_samples))])
print(f"Mean: {comp_scores_baseline.mean():.4f}")
print(f"Std: {comp_scores_baseline.std():.4f}")

hist(comp_scores_baseline, title="Random composition scores", nbins=50, width=800,
     labels={"x": "Composition score"})

In [ ]:
# Cell 50: Composition scores with baseline subtracted
baseline = comp_scores_baseline.mean()

for comp_type in ["Q", "K", "V"]:
    fig = imshow(
        composition_scores[comp_type] - baseline,
        labels={"x": "L1 Head", "y": "L0 Head", "color": "Score - baseline"},
        title=f"{comp_type} Composition Scores (baseline-subtracted)",
        text_auto=".2f", width=700, height=600,
    )

### Bonus: Batched Composition Scores with FactoredMatrix

In [ ]:
# Cell 51: Batched composition scores with FactoredMatrix
def get_batched_comp_scores(
    W_As: FactoredMatrix, W_Bs: FactoredMatrix,
) -> Tensor:
    """Vectorized composition scores.
    W_As: FactoredMatrix with shape (*A_idx, in, out)
    W_Bs: FactoredMatrix with shape (*B_idx, in, out)
    Returns: (*A_idx, *B_idx) tensor of composition scores.
    """
    W_As = FactoredMatrix(
        W_As.A.reshape(-1, 1, *W_As.A.shape[-2:]),
        W_As.B.reshape(-1, 1, *W_As.B.shape[-2:]),
    )
    W_Bs = FactoredMatrix(
        W_Bs.A.reshape(1, -1, *W_Bs.A.shape[-2:]),
        W_Bs.B.reshape(1, -1, *W_Bs.B.shape[-2:]),
    )
    W_ABs = W_As @ W_Bs
    return W_ABs.norm() / (W_As.norm() * W_Bs.norm())

In [ ]:
# Cell 52: Verify batched scores match loop-based scores
W_Q_all = torch.stack([raw_model.blocks[l].attn.W_Q for l in range(cfg.n_layers)])
W_K_all = torch.stack([raw_model.blocks[l].attn.W_K for l in range(cfg.n_layers)])
W_V_all = torch.stack([raw_model.blocks[l].attn.W_V for l in range(cfg.n_layers)])
W_O_all = torch.stack([raw_model.blocks[l].attn.W_O for l in range(cfg.n_layers)])

W_QK = FactoredMatrix(W_Q_all, W_K_all.transpose(-1, -2))
W_OV = FactoredMatrix(W_V_all, W_O_all)

composition_scores_batched = {
    "Q": get_batched_comp_scores(W_OV[0], W_QK[1]),
    "K": get_batched_comp_scores(W_OV[0], W_QK[1].T),
    "V": get_batched_comp_scores(W_OV[0], W_OV[1]),
}

for comp_type in ["Q", "K", "V"]:
    torch.testing.assert_close(
        composition_scores_batched[comp_type],
        composition_scores[comp_type],
        atol=1e-4, rtol=1e-3,
    )
print("Batched composition scores match!")

### Targeted Ablation to Verify the Circuit

Ablate each L0 head's value output and measure the effect on L1H4's induction score.

In [ ]:
# Cell 53: Targeted ablation to verify the circuit
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)


def ablation_induction_score(
    prev_head_index: Optional[int], ind_head_index: int,
) -> float:
    """Ablate a specific L0 head (set its V output to 0) and measure
    the induction score of the specified L1 head."""
    with model.trace(rep_tokens):
        if prev_head_index is not None:
            # Direct modification on proxy - cleaner nnsight pattern
            model.blocks[0].attn.hook_v.output[:, :, prev_head_index, :] = 0.0
        pattern = model.blocks[1].attn.hook_pattern.output.save()

    # Extract induction score for the specified head
    return pattern[0, ind_head_index].diag(-(seq_len - 1)).mean().item()


baseline_score = ablation_induction_score(None, 4)
print(f"Induction score for no ablations: {baseline_score:.5f}\n")

for i in range(cfg.n_heads):
    new_score = ablation_induction_score(i, 4)
    change = new_score - baseline_score
    print(f"Ablation score change for head {i:02}: {change:+.5f}")

---
## Summary

**Key nnsight patterns used in this notebook:**

| TransformerLens | nnsight equivalent |
|---|---|
| `logits, cache = model.run_with_cache(tokens)` | `with model.trace(tokens): val = model.module.output.save()` |
| `cache["pattern", layer]` | `model.blocks[layer].attn.hook_pattern.output.save()` |
| `cache["result", layer]` | `model.blocks[layer].attn.hook_result.output.save()` |
| `model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` | Modify outputs directly inside `model.trace()` |
| `z[:, :, head, :] = 0.0` (in hook) | Same code, inside `with model.trace():` |
| `model.W_Q[layer, head]` | `raw_model.blocks[layer].attn.W_Q[head]` |